In [2]:
import pandas as pd
import numpy as np
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [3]:
df_full = pd.read_csv("005_manchester_processed.csv")
df_full = df_full.sort_values(by=['year','month','day'])

df = df_full.drop(columns=['lat','lon','time'], errors='ignore')

target = "TREFMXAV_U"

In [4]:
#split dataset
train_df = df[(df['year'] >= 2006) & (df['year'] <= 2040)].copy()
test_df  = df[(df['year'] > 2040) & (df['year'] < 2050)].copy()


train_df = train_df.sort_values(by=['year','month','day']).reset_index(drop=True)
test_df  = test_df.sort_values(by=['year','month','day']).reset_index(drop=True)


In [5]:
#h2o
h2o.init(max_mem_size="4G")

train_h2o = h2o.H2OFrame(train_df)
test_h2o  = h2o.H2OFrame(test_df)

features = [col for col in train_df.columns if col != target]

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,6 days 23 hours 9 mins
H2O_cluster_timezone:,Europe/London
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,16 days
H2O_cluster_name:,H2O_from_python_jessie_ruxrz6
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.101 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [6]:
#AutoML training
aml = H2OAutoML(
    max_models=10,
    seed=42,
    max_runtime_secs=600
)

aml.train(
    x=features,
    y=target,
    training_frame=train_h2o,
    leaderboard_frame=test_h2o
)


AutoML progress: |
22:44:55.216: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),8/10
# GBM base models (used / total),6/6
# DRF base models (used / total),1/2
# GLM base models (used / total),0/1
# DeepLearning base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [7]:
#test
preds = aml.leader.predict(test_h2o)
preds_df = preds.as_data_frame().reset_index(drop=True)

# avoid index wrong
test_df['pred'] = preds_df.iloc[:, 0].values

stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


/opt/anaconda3/lib/python3.13/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [10]:
#evaluation
mask = test_df[target].notna() & test_df['pred'].notna()

y_true = test_df.loc[mask, target]
y_pred = test_df.loc[mask, 'pred']

mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred) ** 0.5

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.517293313740369
RMSE: 0.7014177623001466


In [11]:
#predict
future_raw = df_full[(df_full['year'] >= 2050) & (df_full['year'] <= 2080)].copy()
future_df = future_raw.drop(columns=['lat','lon','time'], errors='ignore')

#future_df = future_df.fillna(method='ffill')

future_df = future_df.reset_index(drop=True)
future_h2o = h2o.H2OFrame(future_df)

future_preds = aml.leader.predict(future_h2o)
future_preds_df = future_preds.as_data_frame().reset_index(drop=True)

future_df['pred'] = future_preds_df.iloc[:, 0].values

print(future_df[['year','month','day','pred']].head())

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
   year  month  day       pred
0  2050      1    1  10.237541
1  2050      1    2  11.899759
2  2050      1    3  10.449775
3  2050      1    4   7.124471
4  2050      1    5   7.682379


/opt/anaconda3/lib/python3.13/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
